# Part 2 — Classification, Impact Simulation, and Model Interpretability

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Cleaning and Feature Engineering

In [2]:
# Load the dataset
df = pd.read_csv("../data/raw/home_credit/application_train.csv")

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

# Top 15 columns with most missing values
df.isnull().sum().sort_values(ascending=False).head(15)

(307511, 122)
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONA

COMMONAREA_AVG              214865
COMMONAREA_MODE             214865
COMMONAREA_MEDI             214865
NONLIVINGAPARTMENTS_MEDI    213514
NONLIVINGAPARTMENTS_MODE    213514
NONLIVINGAPARTMENTS_AVG     213514
FONDKAPREMONT_MODE          210295
LIVINGAPARTMENTS_AVG        210199
LIVINGAPARTMENTS_MEDI       210199
LIVINGAPARTMENTS_MODE       210199
FLOORSMIN_MODE              208642
FLOORSMIN_AVG               208642
FLOORSMIN_MEDI              208642
YEARS_BUILD_AVG             204488
YEARS_BUILD_MODE            204488
dtype: int64

### Cleaning audit setup
We record before/after counts for every required cleaning rule:
- columns dropped for >40% missingness
- numeric median imputation
- categorical mode imputation
- extreme income outlier removal
- anomalous `DAYS_EMPLOYED` removal

In [3]:
# Keep an audit log for the report
cleaning_audit = {}

# Snapshot before cleaning
rows_before = df.shape[0]
cols_before = df.shape[1]

print(f"Initial shape: {df.shape}")

Initial shape: (307511, 122)


In [4]:
# Rule 2a: Drop columns with >40% missing values
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 40].sort_values(ascending=False).index.tolist()

cleaning_audit["drop_cols_gt_40pct_missing"] = {
    "num_columns_dropped": len(cols_to_drop),
    "columns_dropped": cols_to_drop
}

print(f"Columns to drop (>40% missing): {len(cols_to_drop)}")
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)

print(f"Shape after dropping high-missing columns: {df.shape}")

Columns to drop (>40% missing): 49
['COMMONAREA_AVG', 'COMMONAREA_MEDI', 'COMMONAREA_MODE', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MODE', 'FONDKAPREMONT_MODE', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MODE', 'FLOORSMIN_MODE', 'FLOORSMIN_AVG', 'FLOORSMIN_MEDI', 'YEARS_BUILD_MODE', 'YEARS_BUILD_MEDI', 'YEARS_BUILD_AVG', 'OWN_CAR_AGE', 'LANDAREA_MODE', 'LANDAREA_MEDI', 'LANDAREA_AVG', 'BASEMENTAREA_AVG', 'BASEMENTAREA_MODE', 'BASEMENTAREA_MEDI', 'EXT_SOURCE_1', 'NONLIVINGAREA_AVG', 'NONLIVINGAREA_MEDI', 'NONLIVINGAREA_MODE', 'ELEVATORS_MODE', 'ELEVATORS_MEDI', 'ELEVATORS_AVG', 'WALLSMATERIAL_MODE', 'APARTMENTS_MODE', 'APARTMENTS_AVG', 'APARTMENTS_MEDI', 'ENTRANCES_MODE', 'ENTRANCES_MEDI', 'ENTRANCES_AVG', 'LIVINGAREA_MEDI', 'LIVINGAREA_MODE', 'LIVINGAREA_AVG', 'HOUSETYPE_MODE', 'FLOORSMAX_MODE', 'FLOORSMAX_AVG', 'FLOORSMAX_MEDI', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BEGINEXPLUATATION_MODE', 'TO

### Numeric missing-value imputation
For all remaining numeric columns with missing values, impute using the median. 

In [5]:
# Rule 2b: Median imputation for remaining numeric columns
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

# Exclude TARGET from imputation logic if you want to be explicit
numeric_feature_cols = [c for c in numeric_cols if c != "TARGET"]

numeric_missing_before = df[numeric_feature_cols].isnull().sum()
numeric_missing_cols = numeric_missing_before[numeric_missing_before > 0].index.tolist()
numeric_missing_total_before = int(numeric_missing_before.sum())

median_impute_summary = []

for col in numeric_missing_cols:
    before_missing = int(df[col].isnull().sum())
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    after_missing = int(df[col].isnull().sum())
    median_impute_summary.append({
        "column": col,
        "median_used": median_value,
        "missing_before": before_missing,
        "missing_after": after_missing
    })

numeric_missing_total_after = int(df[numeric_feature_cols].isnull().sum().sum())

cleaning_audit["numeric_median_imputation"] = {
    "num_columns_imputed": len(numeric_missing_cols),
    "total_missing_before": numeric_missing_total_before,
    "total_missing_after": numeric_missing_total_after
}

print(f"Numeric columns imputed with median: {len(numeric_missing_cols)}")
print(f"Total numeric missing values before: {numeric_missing_total_before}")
print(f"Total numeric missing values after: {numeric_missing_total_after}")

pd.DataFrame(median_impute_summary).head(20)

Numeric columns imputed with median: 16
Total numeric missing values before: 315116
Total numeric missing values after: 0


,column,median_used,missing_before,missing_after
0,AMT_ANNUITY,24903.000000,12,0
1,AMT_GOODS_PRICE,450000.000000,278,0
2,CNT_FAM_MEMBERS,2.000000,2,0
3,EXT_SOURCE_2,0.565961,660,0
4,EXT_SOURCE_3,0.535276,60965,0
5,OBS_30_CNT_SOCIAL_CIRCLE,0.000000,1021,0
6,DEF_30_CNT_SOCIAL_CIRCLE,0.000000,1021,0
7,OBS_60_CNT_SOCIAL_CIRCLE,0.000000,1021,0
8,DEF_60_CNT_SOCIAL_CIRCLE,0.000000,1021,0
9,DAYS_LAST_PHONE_CHANGE,-757.000000,1,0


In [6]:
# Rule 2c: Mode imputation for remaining categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

cat_missing_before = df[categorical_cols].isnull().sum() if categorical_cols else pd.Series(dtype=int)
cat_missing_cols = cat_missing_before[cat_missing_before > 0].index.tolist()
cat_missing_total_before = int(cat_missing_before.sum()) if len(cat_missing_before) else 0

mode_impute_summary = []

for col in cat_missing_cols:
    before_missing = int(df[col].isnull().sum())
    mode_value = df[col].mode(dropna=True)[0]
    df[col] = df[col].fillna(mode_value)
    after_missing = int(df[col].isnull().sum())
    mode_impute_summary.append({
        "column": col,
        "mode_used": mode_value,
        "missing_before": before_missing,
        "missing_after": after_missing
    })

cat_missing_total_after = int(df[categorical_cols].isnull().sum().sum()) if categorical_cols else 0

cleaning_audit["categorical_mode_imputation"] = {
    "num_columns_imputed": len(cat_missing_cols),
    "total_missing_before": cat_missing_total_before,
    "total_missing_after": cat_missing_total_after
}

print(f"Categorical columns imputed with mode: {len(cat_missing_cols)}")
print(f"Total categorical missing values before: {cat_missing_total_before}")
print(f"Total categorical missing values after: {cat_missing_total_after}")

pd.DataFrame(mode_impute_summary).head(20)

Categorical columns imputed with mode: 2
Total categorical missing values before: 97683
Total categorical missing values after: 0


,column,mode_used,missing_before,missing_after
0,NAME_TYPE_SUITE,Unaccompanied,1292,0
1,OCCUPATION_TYPE,Laborers,96391,0


In [7]:
# Rule 2d: Remove rows where AMT_INCOME_TOTAL is an extreme outlier (>99.5th percentile)
income_threshold = df["AMT_INCOME_TOTAL"].quantile(0.995)
rows_before_income_filter = df.shape[0]

df = df[df["AMT_INCOME_TOTAL"] <= income_threshold].copy()

rows_after_income_filter = df.shape[0]
rows_removed_income = rows_before_income_filter - rows_after_income_filter

cleaning_audit["income_outlier_removal"] = {
    "threshold_99_5pct": float(income_threshold),
    "rows_before": rows_before_income_filter,
    "rows_after": rows_after_income_filter,
    "rows_removed": rows_removed_income
}

print(f"Income outlier threshold (99.5th percentile): {income_threshold:,.2f}")
print(f"Rows removed for AMT_INCOME_TOTAL outliers: {rows_removed_income}")
print(f"Shape after income outlier removal: {df.shape}")

Income outlier threshold (99.5th percentile): 630,000.00
Rows removed for AMT_INCOME_TOTAL outliers: 1446
Shape after income outlier removal: (306065, 73)


In [8]:
# Rule 2e: Remove rows where DAYS_EMPLOYED == 365243
rows_before_days_emp = df.shape[0]

anomalous_days_employed_mask = df["DAYS_EMPLOYED"] == 365243
rows_removed_days_emp = int(anomalous_days_employed_mask.sum())

df = df.loc[~anomalous_days_employed_mask].copy()

rows_after_days_emp = df.shape[0]

cleaning_audit["days_employed_anomaly_removal"] = {
    "anomalous_value": 365243,
    "rows_before": rows_before_days_emp,
    "rows_after": rows_after_days_emp,
    "rows_removed": rows_removed_days_emp
}

print(f"Rows removed where DAYS_EMPLOYED == 365243: {rows_removed_days_emp}")
print(f"Shape after DAYS_EMPLOYED anomaly removal: {df.shape}")

Rows removed where DAYS_EMPLOYED == 365243: 55290
Shape after DAYS_EMPLOYED anomaly removal: (250775, 73)


In [9]:
#Feature engineering
df['credit_income_ratio'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['annuity_income_ratio'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['credit_term'] = df['AMT_CREDIT'] / df['AMT_ANNUITY']
df['days_employed_ratio'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['income_per_family'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

In [10]:
# Re-impute any numeric NaNs introduced by ratio calculations
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
numeric_feature_cols = [c for c in numeric_cols if c != "TARGET"]
for col in numeric_feature_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

In [11]:
# Encode categorical variables using one-hot encoding
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

cleaning_audit["encoding"] = {
    "num_categorical_columns_encoded": len(categorical_cols),
    "shape_before_encoding": tuple(df.shape),
    "shape_after_encoding": tuple(df_encoded.shape)
}

print(f"Categorical columns encoded: {len(categorical_cols)}")
print(f"Shape before encoding: {df.shape}")
print(f"Shape after encoding: {df_encoded.shape}")

Categorical columns encoded: 12
Shape before encoding: (250775, 78)
Shape after encoding: (250775, 176)


### Final validation before saving curated dataset
Confirm there are no remaining missing values in the curated dataset.

In [12]:
remaining_nulls = df_encoded.isnull().sum().sort_values(ascending=False)
remaining_nulls = remaining_nulls[remaining_nulls > 0]

print(f"Remaining columns with nulls: {len(remaining_nulls)}")
if len(remaining_nulls) > 0:
    print(remaining_nulls.head(20))
else:
    print("No remaining missing values in curated dataset.")

Remaining columns with nulls: 0
No remaining missing values in curated dataset.


In [13]:
summary_rows = [
    {
        "step": "Initial dataset",
        "details": f"rows={rows_before}, cols={cols_before}"
    },
    {
        "step": "Drop columns >40% missing",
        "details": f"{cleaning_audit['drop_cols_gt_40pct_missing']['num_columns_dropped']} columns dropped"
    },
    {
        "step": "Numeric median imputation",
        "details": (
            f"{cleaning_audit['numeric_median_imputation']['num_columns_imputed']} columns imputed, "
            f"{cleaning_audit['numeric_median_imputation']['total_missing_before']} -> "
            f"{cleaning_audit['numeric_median_imputation']['total_missing_after']} missing values"
        )
    },
    {
        "step": "Categorical mode imputation",
        "details": (
            f"{cleaning_audit['categorical_mode_imputation']['num_columns_imputed']} columns imputed, "
            f"{cleaning_audit['categorical_mode_imputation']['total_missing_before']} -> "
            f"{cleaning_audit['categorical_mode_imputation']['total_missing_after']} missing values"
        )
    },
    {
        "step": "Income outlier removal",
        "details": f"{cleaning_audit['income_outlier_removal']['rows_removed']} rows removed"
    },
    {
        "step": "DAYS_EMPLOYED anomaly removal",
        "details": f"{cleaning_audit['days_employed_anomaly_removal']['rows_removed']} rows removed"
    },
    {
        "step": "Encoding",
        "details": (
            f"{cleaning_audit['encoding']['num_categorical_columns_encoded']} categorical columns encoded; "
            f"shape {cleaning_audit['encoding']['shape_before_encoding']} -> "
            f"{cleaning_audit['encoding']['shape_after_encoding']}"
        )
    }
]

cleaning_summary_df = pd.DataFrame(summary_rows)
cleaning_summary_df

,step,details
0,Initial dataset,"rows=307511, cols=122"
1,Drop columns >40% missing,49 columns dropped
2,Numeric median imputation,"16 columns imputed, 315116 -> 0 missing values"
3,Categorical mode imputation,"2 columns imputed, 97683 -> 0 missing values"
4,Income outlier removal,1446 rows removed
5,DAYS_EMPLOYED anomaly removal,55290 rows removed
6,Encoding,"12 categorical columns encoded; shape (250775,..."


In [16]:
#Save curated dataset as parquet
df_encoded.to_parquet("../data/curated/home_credit_curated.parquet")

print(f"Saved curated dataset to: ../data/curated/home_credit_curated.parquet")
print(f"Final curated shape: {df_encoded.shape}")

Saved curated dataset to: ../data/curated/home_credit_curated.parquet
Final curated shape: (250775, 176)
